In [1]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog

import pandas as pd

/Users/devan/Documents/nba-performance-predictor/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Understand data available for one specific player (Jalen Brunson), and begin with first stages of feature engineering

In [2]:
player_results = players.find_players_by_full_name("Jalen Brunson")
player_results

[{'id': 1628973,
  'full_name': 'Jalen Brunson',
  'first_name': 'Jalen',
  'last_name': 'Brunson',
  'is_active': True}]

In [3]:
brunson_id = player_results[0]["id"]

game_log = playergamelog.PlayerGameLog(
    player_id=brunson_id,
    season="2025-26"
)

brunson_games = game_log.get_data_frames()[0]

brunson_games.head()

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22025,1628973,0022501176,"Apr 10, 2026",NYK vs. TOR,W,31,12,18,0.667,...,3,3,2,2,0,3,1,29,4,1
1,22025,1628973,0022501168,"Apr 09, 2026",NYK vs. BOS,W,37,10,19,0.526,...,1,1,10,0,0,1,3,25,8,1
2,22025,1628973,0022501143,"Apr 06, 2026",NYK @ ATL,W,39,11,26,0.423,...,3,3,13,2,0,3,1,30,7,1
3,22025,1628973,0022501123,"Apr 03, 2026",NYK vs. CHI,W,30,6,13,0.462,...,1,1,10,0,0,2,4,17,25,1
4,22025,1628973,0022501102,"Mar 31, 2026",NYK @ HOU,L,37,5,14,0.357,...,5,6,8,1,0,3,2,12,-26,1


In [4]:
brunson_games.shape

(74, 27)

In [5]:
brunson_games.columns.tolist()

['SEASON_ID',
 'Player_ID',
 'Game_ID',
 'GAME_DATE',
 'MATCHUP',
 'WL',
 'MIN',
 'FGM',
 'FGA',
 'FG_PCT',
 'FG3M',
 'FG3A',
 'FG3_PCT',
 'FTM',
 'FTA',
 'FT_PCT',
 'OREB',
 'DREB',
 'REB',
 'AST',
 'STL',
 'BLK',
 'TOV',
 'PF',
 'PTS',
 'PLUS_MINUS',
 'VIDEO_AVAILABLE']

In [6]:
brunson_games = brunson_games.sort_values("GAME_DATE")

brunson_games.head()

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
3,22025,1628973,0022501123,"Apr 03, 2026",NYK vs. CHI,W,30,6,13,0.462,...,1,1,10,0,0,2,4,17,25,1
2,22025,1628973,0022501143,"Apr 06, 2026",NYK @ ATL,W,39,11,26,0.423,...,3,3,13,2,0,3,1,30,7,1
1,22025,1628973,0022501168,"Apr 09, 2026",NYK vs. BOS,W,37,10,19,0.526,...,1,1,10,0,0,1,3,25,8,1
0,22025,1628973,0022501176,"Apr 10, 2026",NYK vs. TOR,W,31,12,18,0.667,...,3,3,2,2,0,3,1,29,4,1
56,22025,1628973,0022500320,"Dec 02, 2025",NYK @ BOS,L,39,6,21,0.286,...,2,2,11,1,0,3,2,15,-4,1


In [7]:
brunson_games["GAME_DATE"] = pd.to_datetime(brunson_games["GAME_DATE"])

In [8]:
brunson_games = brunson_games.sort_values("GAME_DATE").reset_index(drop=True)

In [9]:
brunson_games["LAST_GAME_PTS"] = brunson_games["PTS"].shift(1)

brunson_games[
    ["GAME_DATE", "MATCHUP", "PTS", "LAST_GAME_PTS"]
].head(10)

,GAME_DATE,MATCHUP,PTS,LAST_GAME_PTS
0,2025-10-22,NYK vs. CLE,23,NaN
1,2025-10-24,NYK vs. BOS,31,23.0
2,2025-10-26,NYK @ MIA,37,31.0
3,2025-10-28,NYK @ MIL,36,37.0
4,2025-10-31,NYK @ CHI,29,36.0
5,2025-11-02,NYK vs. CHI,31,29.0
6,2025-11-03,NYK vs. WAS,16,31.0
7,2025-11-05,NYK vs. MIN,23,16.0
8,2025-11-09,NYK vs. BKN,19,23.0
9,2025-11-11,NYK vs. MEM,32,19.0


In [10]:
for stat in ["PTS", "REB", "AST"]:
    brunson_games[f"{stat}_LAST_5_AVG"] = (
        brunson_games[stat]
        .shift(1)
        .rolling(window=5)
        .mean()
    )

    brunson_games[f"{stat}_LAST_10_AVG"] = (
        brunson_games[stat]
        .shift(1)
        .rolling(window=10)
        .mean()
    )

In [11]:
brunson_games[
    [
        "GAME_DATE",
        "PTS",
        "PTS_LAST_5_AVG",
        "PTS_LAST_10_AVG",
        "REB_LAST_5_AVG",
        "AST_LAST_5_AVG",
    ]
].head(15)

,GAME_DATE,PTS,PTS_LAST_5_AVG,PTS_LAST_10_AVG,REB_LAST_5_AVG,AST_LAST_5_AVG
0,2025-10-22,23,NaN,NaN,NaN,NaN
1,2025-10-24,31,NaN,NaN,NaN,NaN
2,2025-10-26,37,NaN,NaN,NaN,NaN
3,2025-10-28,36,NaN,NaN,NaN,NaN
4,2025-10-31,29,NaN,NaN,NaN,NaN
5,2025-11-02,31,31.2,NaN,3.6,5.4
6,2025-11-03,16,32.8,NaN,3.8,5.0
7,2025-11-05,23,29.8,NaN,3.4,5.8
8,2025-11-09,19,27.0,NaN,3.8,6.4
9,2025-11-11,32,23.6,NaN,3.0,7.2


## Breakdown top active players

In [16]:
from nba_api.stats.static import players

all_players = players.get_players()

len(all_players)

5135

In [17]:
all_players[:5]

[{'id': 76001,
  'full_name': 'Alaa Abdelnaby',
  'first_name': 'Alaa',
  'last_name': 'Abdelnaby',
  'is_active': False},
 {'id': 76002,
  'full_name': 'Zaid Abdul-Aziz',
  'first_name': 'Zaid',
  'last_name': 'Abdul-Aziz',
  'is_active': False},
 {'id': 76003,
  'full_name': 'Kareem Abdul-Jabbar',
  'first_name': 'Kareem',
  'last_name': 'Abdul-Jabbar',
  'is_active': False},
 {'id': 51,
  'full_name': 'Mahmoud Abdul-Rauf',
  'first_name': 'Mahmoud',
  'last_name': 'Abdul-Rauf',
  'is_active': False},
 {'id': 1505,
  'full_name': 'Tariq Abdul-Wahad',
  'first_name': 'Tariq',
  'last_name': 'Abdul-Wahad',
  'is_active': False}]

In [18]:
active_players = [player for player in all_players if player["is_active"]]

len(active_players)

571

In [20]:
active_players[:10]

[{'id': 1630173,
  'full_name': 'Precious Achiuwa',
  'first_name': 'Precious',
  'last_name': 'Achiuwa',
  'is_active': True},
 {'id': 203500,
  'full_name': 'Steven Adams',
  'first_name': 'Steven',
  'last_name': 'Adams',
  'is_active': True},
 {'id': 1628389,
  'full_name': 'Bam Adebayo',
  'first_name': 'Bam',
  'last_name': 'Adebayo',
  'is_active': True},
 {'id': 1630534,
  'full_name': 'Ochai Agbaji',
  'first_name': 'Ochai',
  'last_name': 'Agbaji',
  'is_active': True},
 {'id': 1630583,
  'full_name': 'Santi Aldama',
  'first_name': 'Santi',
  'last_name': 'Aldama',
  'is_active': True},
 {'id': 1641725,
  'full_name': 'Trey Alexander',
  'first_name': 'Trey',
  'last_name': 'Alexander',
  'is_active': True},
 {'id': 1629638,
  'full_name': 'Nickeil Alexander-Walker',
  'first_name': 'Nickeil',
  'last_name': 'Alexander-Walker',
  'is_active': True},
 {'id': 1628960,
  'full_name': 'Grayson Allen',
  'first_name': 'Grayson',
  'last_name': 'Allen',
  'is_active': True},
 {'id

In [22]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.data_processing import get_player_game_logs

test_games = get_player_game_logs(
    player_id=1628973,
    season="2025-26"
)

test_games[["GAME_DATE", "MATCHUP", "PTS", "REB", "AST"]].head()

,GAME_DATE,MATCHUP,PTS,REB,AST
0,2025-10-22,NYK vs. CLE,23,4,5
1,2025-10-24,NYK vs. BOS,31,3,5
2,2025-10-26,NYK @ MIA,37,5,7
3,2025-10-28,NYK @ MIL,36,4,3
4,2025-10-31,NYK @ CHI,29,2,7


In [27]:
from importlib import reload
import src.data_processing

reload(src.data_processing)

from src.data_processing import engineer_features

brunson_features = engineer_features(test_games)

brunson_features[
    [
        "GAME_DATE",
        "MATCHUP",
        "OPPONENT",
        "IS_HOME",
        "REST_DAYS",
        "PTS",
        "PTS_LAST_5_AVG",
        "PTS_LAST_10_AVG",
        "PTS_SEASON_AVG",
    ]
].head(12)

,GAME_DATE,MATCHUP,OPPONENT,IS_HOME,REST_DAYS,PTS,PTS_LAST_5_AVG,PTS_LAST_10_AVG,PTS_SEASON_AVG
0,2025-10-22,NYK vs. CLE,CLE,1,NaN,23,NaN,NaN,NaN
1,2025-10-24,NYK vs. BOS,BOS,1,2.0,31,NaN,NaN,23.000000
2,2025-10-26,NYK @ MIA,MIA,0,2.0,37,NaN,NaN,27.000000
3,2025-10-28,NYK @ MIL,MIL,0,2.0,36,NaN,NaN,30.333333
4,2025-10-31,NYK @ CHI,CHI,0,3.0,29,NaN,NaN,31.750000
5,2025-11-02,NYK vs. CHI,CHI,1,2.0,31,31.2,NaN,31.200000
6,2025-11-03,NYK vs. WAS,WAS,1,1.0,16,32.8,NaN,31.166667
7,2025-11-05,NYK vs. MIN,MIN,1,2.0,23,29.8,NaN,29.000000
8,2025-11-09,NYK vs. BKN,BKN,1,4.0,19,27.0,NaN,28.250000
9,2025-11-11,NYK vs. MEM,MEM,1,2.0,32,23.6,NaN,27.222222


In [31]:
from importlib import reload
import src.data_processing

reload(src.data_processing)

from src.data_processing import build_player_dataset

brunson_dataset = build_player_dataset(
    player_id=1628973,
    season="2025-26"
)

brunson_dataset.head(12)

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,PTS_LAST_10_AVG,PTS_SEASON_AVG,REB_LAST_GAME,REB_LAST_5_AVG,REB_LAST_10_AVG,REB_SEASON_AVG,AST_LAST_GAME,AST_LAST_5_AVG,AST_LAST_10_AVG,AST_SEASON_AVG
0,22025,1628973,0022500003,2025-10-22,NYK vs. CLE,W,34,5,18,0.278,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22025,1628973,0022500018,2025-10-24,NYK vs. BOS,W,35,10,20,0.500,...,NaN,23.000000,4.0,NaN,NaN,4.000000,5.0,NaN,NaN,5.000000
2,22025,1628973,0022500108,2025-10-26,NYK @ MIA,L,35,14,26,0.538,...,NaN,27.000000,3.0,NaN,NaN,3.500000,5.0,NaN,NaN,5.000000
3,22025,1628973,0022500125,2025-10-28,NYK @ MIL,L,35,14,25,0.560,...,NaN,30.333333,5.0,NaN,NaN,4.000000,7.0,NaN,NaN,5.666667
4,22025,1628973,0022500023,2025-10-31,NYK @ CHI,L,35,12,25,0.480,...,NaN,31.750000,4.0,NaN,NaN,4.000000,3.0,NaN,NaN,5.000000
5,22025,1628973,0022500153,2025-11-02,NYK vs. CHI,W,32,10,22,0.455,...,NaN,31.200000,2.0,3.6,NaN,3.600000,7.0,5.4,NaN,5.400000
6,22025,1628973,0022500159,2025-11-03,NYK vs. WAS,W,32,6,17,0.353,...,NaN,31.166667,5.0,3.8,NaN,3.833333,3.0,5.0,NaN,5.000000
7,22025,1628973,0022500175,2025-11-05,NYK vs. MIN,W,33,9,20,0.450,...,NaN,29.000000,1.0,3.4,NaN,3.428571,9.0,5.8,NaN,5.571429
8,22025,1628973,0022500192,2025-11-09,NYK vs. BKN,W,29,6,14,0.429,...,NaN,28.250000,7.0,3.8,NaN,3.875000,10.0,6.4,NaN,6.125000
9,22025,1628973,0022500208,2025-11-11,NYK vs. MEM,W,36,11,19,0.579,...,NaN,27.222222,0.0,3.0,NaN,3.444444,7.0,7.2,NaN,6.222222
